In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
# --- 1. Data Loading (The Torchvision Way) ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST normalization
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

100%|██████████| 9.91M/9.91M [00:13<00:00, 732kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.90MB/s]


In [3]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import numpy as np

In [32]:
class MINSTmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(28*28, 128)
        self.layer2 = nn.Linear(128, 10)
    def __call__(self, x):
        x = mx.flatten(x, start_axis=1) # Flatten the input
        x = self.layer1(x)
        x = nn.relu(x)
        x = self.layer2(x)
        return x

In [37]:
data, target = next(iter(train_loader))
print(f"Data Max: {data.max()}, Data Min: {data.min()}") 
# It should be roughly 2.8 and -0.4 due to the Normalize((0.1307,), (0.3081,))

Data Max: 2.821486711502075, Data Min: -0.4242129623889923


In [33]:
model = MINSTmodel()
mx.eval(model.parameters())

In [34]:
def loss_fn(model, X, y): # Needs 3 arguments
    logits = model(X)
    return mx.mean(nn.losses.cross_entropy(logits, y))

In [35]:
optimizer = optim.Adam(learning_rate=1e-3)
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

In [36]:
@mx.compile
def train_step(X, y):
    loss, grads = loss_and_grad_fn(model, X, y)
    optimizer.update(model, grads)
    return loss

# --- 4. Training Loop ---
for epoch in range(3):
    for batch_idx, (data, target) in enumerate(train_loader):
        # The key bridge: Torch Tensor -> NumPy -> MLX Array
        X = mx.array(data.numpy())
        y = mx.array(target.numpy())

        loss = train_step(X, y)
        
        if batch_idx % 100 == 0:
            mx.eval(loss) # Force computation to see the value
            print(f"Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f}")

print("Training Complete!")

Epoch 0 | Batch 0 | Loss: 2.3902
Epoch 0 | Batch 100 | Loss: 2.2974
Epoch 0 | Batch 200 | Loss: 2.3697
Epoch 0 | Batch 300 | Loss: 2.3737
Epoch 0 | Batch 400 | Loss: 2.3418
Epoch 0 | Batch 500 | Loss: 2.3474
Epoch 0 | Batch 600 | Loss: 2.3944
Epoch 0 | Batch 700 | Loss: 2.3750
Epoch 0 | Batch 800 | Loss: 2.3865
Epoch 0 | Batch 900 | Loss: 2.3034
Epoch 1 | Batch 0 | Loss: 2.3160
Epoch 1 | Batch 100 | Loss: 2.3755
Epoch 1 | Batch 200 | Loss: 2.3948
Epoch 1 | Batch 300 | Loss: 2.3671
Epoch 1 | Batch 400 | Loss: 2.3292
Epoch 1 | Batch 500 | Loss: 2.3407
Epoch 1 | Batch 600 | Loss: 2.3519
Epoch 1 | Batch 700 | Loss: 2.3307
Epoch 1 | Batch 800 | Loss: 2.3375
Epoch 1 | Batch 900 | Loss: 2.3591
Epoch 2 | Batch 0 | Loss: 2.3989
Epoch 2 | Batch 100 | Loss: 2.3427
Epoch 2 | Batch 200 | Loss: 2.3828
Epoch 2 | Batch 300 | Loss: 2.3243
Epoch 2 | Batch 400 | Loss: 2.3432
Epoch 2 | Batch 500 | Loss: 2.3713
Epoch 2 | Batch 600 | Loss: 2.3768
Epoch 2 | Batch 700 | Loss: 2.3443
Epoch 2 | Batch 800 | Loss

In [38]:
# Debug: Print shapes and types before training step
print(f"Batch {batch_idx}: X shape: {X.shape}, X dtype: {X.dtype}, y shape: {y.shape}, y dtype: {y.dtype}")
# Debug: Print model output stats
logits = model(X)
print(f"Logits min: {logits.min()}, max: {logits.max()}, mean: {logits.mean()}")
# Debug: Print unique labels in y
print(f"Unique labels in y: {np.unique(y)}")

Batch 937: X shape: (32, 1, 28, 28), X dtype: mlx.core.float32, y shape: (32,), y dtype: mlx.core.int64


RuntimeError: [eval] Attempting to eval an array without a primitive.
If you are compiling a function, make sure all the inputs and outputs are captured:
https://ml-explore.github.io/mlx/build/html/usage/compile.html#pure-functions.
If you are not using compile, this may be a bug. Please file an issue here:
https://github.com/ml-explore/mlx/issues.

In [45]:
a = np.array([[3, -4], [4, 3]])
t = np.array([[2, 0], [0, 1]])

In [48]:
t

array([[2, 0],
       [0, 1]])

In [50]:
np.dot(a, np.dot(t, np.linalg.inv(a)))

array([[1.36, 0.48],
       [0.48, 1.64]])

In [52]:
( a @ t @ np.linalg.inv(a) ) @ np.array([[3], [4]])

array([[6.],
       [8.]])